# Data exploration

This notebook documents the **four** tabular datasets under `Data/`. Each pair (`*_train.csv` / `*_test.csv`) is a separate modeling problem with its own features and target.

| File stem | Rows (train) | Focus |
|-----------|--------------|--------|
| `medical_risk` | 2,000 | Health risk from vitals and lifestyle |
| `hotel_demand` | 2,000 | Booking / demand prediction |
| `complaint_nlp` | 2,000 | Customer complaint text classification |
| `candidate_success` | 2,000 | Hiring / success prediction from profile signals |

The sections below answer, **for each dataset**: (1) description, (2) an E/R-style model, (3) the target variable, and (4) the business problem it represents.


## 1. Medical risk (`medical_risk_train.csv` / `medical_risk_test.csv`)

### Describe the dataset

Each row is one **patient record** identified by `id`. The features are clinical and behavioral: `age`, `bmi`, `blood_pressure`, `cholesterol_level`, `glucose_level`, plus indicators `smoker`, `physical_activity`, `family_history`, and `stress_level` (numeric scale). The training split has **2,000** rows; labels are imbalanced toward the negative class (**1,329** vs **671** positives for `risk_label`).

### E/R model

Conceptually, one entity **PatientRecord** holds measurements and the outcome used for supervised learning. The diagram treats `risk_label` as the prediction target linked to the same record.

```mermaid
erDiagram
    PATIENT_RECORD {
        int id PK
        int age
        float bmi
        int blood_pressure
        int cholesterol_level
        int glucose_level
        int smoker
        int physical_activity
        int family_history
        int stress_level
        int risk_label "target"
    }
```

### What is the target variable?

- **`risk_label`** — binary outcome: **1** = higher modeled health risk, **0** = lower risk (e.g., for prioritization or screening).

### Business problem

A healthcare analytics or insurer use case: **prioritize patients for follow-up, prevention programs, or resource allocation** by predicting elevated risk from routine measurements and questionnaire-style inputs, without replacing clinical diagnosis.


## 2. Hotel demand (`hotel_demand_train.csv` / `hotel_demand_test.csv`)

### Describe the dataset

Each row is one **booking** (or booking request) with operational attributes: `hotel_type`, `lead_time_days`, `stay_length`, `num_adults`, `num_children`, `season`, `price_per_night`, `previous_bookings`, `special_requests`. Training has **2,000** rows; `demand_label` is roughly **1,143** vs **857** between the two classes.

### E/R model

Single entity **Booking** with all reservation-side attributes and the demand outcome.

```mermaid
erDiagram
    BOOKING {
        int id PK
        int hotel_type
        int lead_time_days
        int stay_length
        int num_adults
        int num_children
        int season
        float price_per_night
        int previous_bookings
        int special_requests
        int demand_label "target"
    }
```

### What is the target variable?

- **`demand_label`** — binary label for **demand level** associated with the booking context (e.g., high vs low demand period or intensity; align with your course definition if provided).

### Business problem

**Revenue and operations planning**: forecast whether a booking pattern corresponds to high demand so the hotel can adjust **pricing, staffing, and inventory** (rooms, amenities) ahead of peak or slack periods.


## 3. Customer complaints — NLP (`complaint_nlp_train.csv` / `complaint_nlp_test.csv`)

### Describe the dataset

Each row is one **customer complaint** with free text `complaint_text` and a single categorical **`category_label`**. Training has **2,000** rows; categories are roughly balanced (**about 368–418** rows per class across five categories: `booking`, `billing`, `cleanliness`, `service`, `technical`).

### E/R model

One entity **Complaint** stores the message; the category is the supervised label. In a normalized database, categories would live in a lookup table referenced by foreign key.

```mermaid
erDiagram
    COMPLAINT {
        int id PK
        string complaint_text
        string category_label "target"
    }
    COMPLAINT_CATEGORY {
        string name PK
    }
    COMPLAINT }o--|| COMPLAINT_CATEGORY : references
```

*(In the flat CSV, `category_label` is denormalized text on each row.)*

### What is the target variable?

- **`category_label`** — **multi-class** outcome: which theme or routing bucket the complaint belongs to (`booking`, `billing`, `cleanliness`, `service`, `technical`).

### Business problem

**Automated routing and analytics**: classify inbound complaints so they reach the **right team** quickly, track issue mix over time, and reduce manual triage in support or hospitality operations.


## 4. Candidate success (`candidate_success_train.csv` / `candidate_success_test.csv`)

### Describe the dataset

Each row is one **candidate** (applicant) profile: `experience_years`, `python_skill_score`, `ml_skill_score`, `projects_completed`, `education_level`, `github_activity`, `communication_score`, `certifications`. Training has **2,000** rows; `success_label` is **1,210** vs **790** for the two classes.

### E/R model

Single entity **Candidate** aggregating résumé-like signals and the hiring outcome.

```mermaid
erDiagram
    CANDIDATE {
        int id PK
        int experience_years
        int python_skill_score
        int ml_skill_score
        int projects_completed
        int education_level
        int github_activity
        int communication_score
        int certifications
        int success_label "target"
    }
```

### What is the target variable?

- **`success_label`** — binary outcome indicating **whether the candidate is successful** under the problem definition (e.g., offer extended, passed a hiring bar, or met a success criterion).

### Business problem

**Recruiting and talent analytics**: predict **hire or success potential** from structured skill and activity features to support **screening, interview prioritization, and workforce planning** (with appropriate fairness and governance in real deployments).


## Structural checks and data quality

This part answers, **for each CSV pair** in `Data/`:

- **Shape** — number of rows and columns (train and test).
- **Prediction features** — columns used as inputs (excluding `id` and the target).
- **Data quality** — missing values; duplicate rows (full row and excluding `id`); univariate **numeric outliers** via the IQR rule (values outside \(Q1 - 1.5 \cdot IQR\) or \(Q3 + 1.5 \cdot IQR\)); for `complaint_text`, the same rule applied to **string length** flags unusually short or long messages.

*Note: Many fields are discrete or bounded, so IQR may flag few or no numeric outliers even when values look "extreme" in a business sense.*


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("Data")

TASKS = [
    ("medical_risk", "risk_label"),
    ("hotel_demand", "demand_label"),
    ("complaint_nlp", "category_label"),
    ("candidate_success", "success_label"),
]


def iqr_outlier_mask(s: pd.Series) -> pd.Series:
    """True where numeric value is outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series(False, index=s.index)
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return pd.Series(False, index=s.index)
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (x < lo) | (x > hi)


def report_split(name: str, df: pd.DataFrame, target: str) -> None:
    feat_cols = [c for c in df.columns if c not in ("id", target)]
    print(f"  Rows: {len(df):,}  |  Columns: {df.shape[1]}")
    print(f"  Features for prediction ({len(feat_cols)}): {feat_cols}")

    miss = df.isna().sum()
    if miss.sum() == 0:
        print("  Missing values: none")
    else:
        print(f"  Missing values (total): {int(miss.sum())}")
        print(miss[miss > 0].to_string())

    dup_all = int(df.duplicated().sum())
    cols_no_id = [c for c in df.columns if c != "id"]
    dup_no_id = int(df.duplicated(subset=cols_no_id).sum())
    print(f"  Duplicate rows (all columns): {dup_all}")
    print(f"  Duplicate rows (same feature+target, ignoring id): {dup_no_id}")

    num_feats = [c for c in feat_cols if pd.api.types.is_numeric_dtype(df[c])]
    any_out_idx = set()
    for col in num_feats:
        m = iqr_outlier_mask(df[col])
        if m.any():
            print(f"  IQR outliers in feature {col!r}: {int(m.sum())} cell(s)")
            any_out_idx.update(df.index[m])
    if num_feats and any_out_idx:
        print(f"  Rows with >=1 numeric feature outlier (any column): {len(any_out_idx)}")

    if "complaint_text" in df.columns:
        lens = df["complaint_text"].astype(str).str.len()
        q1, q3 = lens.quantile(0.25), lens.quantile(0.75)
        iqr = q3 - q1
        if iqr > 0:
            lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            mlen = (lens < lo) | (lens > hi)
            if mlen.any():
                print(
                    f"  IQR outliers on complaint_text length: {int(mlen.sum())} row(s)"
                )


for stem, target in TASKS:
    print("=" * 72)
    print(stem)
    print("-" * 72)
    train = pd.read_csv(DATA_DIR / f"{stem}_train.csv")
    test = pd.read_csv(DATA_DIR / f"{stem}_test.csv")
    print("TRAIN")
    report_split(stem, train, target)
    print("TEST")
    report_split(stem, test, target)

summary_rows = []
for stem, target in TASKS:
    tr = pd.read_csv(DATA_DIR / f"{stem}_train.csv")
    te = pd.read_csv(DATA_DIR / f"{stem}_test.csv")
    summary_rows.append(
        {
            "dataset": stem,
            "split": "train",
            "rows": len(tr),
            "columns": tr.shape[1],
        }
    )
    summary_rows.append(
        {
            "dataset": stem,
            "split": "test",
            "rows": len(te),
            "columns": te.shape[1],
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


medical_risk
------------------------------------------------------------------------
TRAIN
  Rows: 2,000  |  Columns: 11
  Features for prediction (9): ['age', 'bmi', 'blood_pressure', 'cholesterol_level', 'glucose_level', 'smoker', 'physical_activity', 'family_history', 'stress_level']
  Missing values: none
  Duplicate rows (all columns): 0
  Duplicate rows (same feature+target, ignoring id): 0
TEST
  Rows: 500  |  Columns: 10
  Features for prediction (9): ['age', 'bmi', 'blood_pressure', 'cholesterol_level', 'glucose_level', 'smoker', 'physical_activity', 'family_history', 'stress_level']
  Missing values: none
  Duplicate rows (all columns): 0
  Duplicate rows (same feature+target, ignoring id): 0
hotel_demand
------------------------------------------------------------------------
TRAIN
  Rows: 2,000  |  Columns: 11
  Features for prediction (9): ['hotel_type', 'lead_time_days', 'stay_length', 'num_adults', 'num_children', 'season', 'price_per_night', 'previous_bookings', 'speci

,dataset,split,rows,columns
0,medical_risk,train,2000,11
1,medical_risk,test,500,10
2,hotel_demand,train,2000,11
3,hotel_demand,test,500,10
4,complaint_nlp,train,2000,3
5,complaint_nlp,test,500,2
6,candidate_success,train,2000,10
7,candidate_success,test,500,9
